## Закрепим знания о предобработке данных на практике

Импортируем необходимые библиотеки

In [ ]:
# import <имя библиотеки> as <как будем её называть в коде>
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

### Этап 1: Загрузка и просмотр данных

Будем использовать набор данных про физическую активность людей

In [ ]:
# Загрузим данные из csv-таблицы
df = pd.read_csv('fitness_data.csv')

In [ ]:
print("Первые 5 строк датасета:")
df.head(5)

In [ ]:
print("Информация о данных:")
df.info()

In [ ]:
print("Пропуски в данных:")
df.isnull().sum()

In [ ]:
print("Основные статистики (число значений, среднее по колонке, мин/макс значения и так далее):")
df.describe()

### Этап 2: Разделение на train и test

In [ ]:
# Целевая переменная - "calories_burned", признаки - все остальные колонки
X = df.drop('calories_burned', axis=1)
y = df['calories_burned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Посмотрим размеры выборок
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

### Этап 3: Обработка пропущенных значений

In [ ]:
# Заполним пропущенные значения в колонке "age" медианным возрастом.

# Считаем медианный возраст
age_median_train = X_train['age'].median()

# Заполняем пропуски в колонке "age" в тренировочной выборке
X_train['age'] = X_train['age'].fillna(age_median_train)

# Заполняем пропуски в колонке "age" в тестовой выборке
# Для этого используем медиану из train, помним про утечку данных из лекции
X_test['age'] = X_test['age'].fillna(age_median_train)

In [ ]:
# По такому же принципу заполните пропуски в остальных числовых колонках: 'weight', 'height', 'resting_bpm', 'birth_year'
# Пишите ваш код в этой ячейке
...

In [ ]:
# Заполним пропуски в колонке "favorite_color" - эт категориальный признак
# Здесь будем использовать самую популярную категорию

train_mode = X_train['favorite_color'].mode()[0]
X_train['favorite_color'] = X_train['favorite_color'].fillna(train_mode)
X_test['favorite_color'] = X_test['favorite_color'].fillna(train_mode)

In [ ]:
# Убедимся, что пропущенных значений больше нет
print(f"Пропусков в тренировочной выборке:\n{X_train.isnull().sum()}\n")
print(f"Пропусков в тестовой выборке:\n{X_test.isnull().sum()}")

### Этап 4: Обработка выбросов

In [ ]:
# Напишем такую функцию, которая принимает датафрейм и имя колонки и удаляет выбросы методом IQR
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

In [ ]:
# Удалим выбросы во всех числовых признаках в X_train
numeric_cols = ['age', 'weight', 'height', 'avg_bpm', 'resting_bpm', 'session_duration'] # Список числовых колонок

print("Размер до обработки выбросов:", X_train.shape)
X_train_clean = X_train.copy()

for col in numeric_cols:
    X_train_clean = remove_outliers_iqr(X_train_clean, col)
    print(f"После обработки {col}: {X_train_clean.shape}")

print("Итоговый размер после обработки выбросов:", X_train_clean.shape)

In [ ]:
# Удалим те же строки из целевой переменной (y_train)
y_train_clean = y_train.loc[X_train_clean.index]
print(f"Размер y_train: {y_train_clean.shape}")

Важно следить за тем, чтобы число строк (объектов) в X_train и y_train было равно

### Этап 5: Создание новых признаков

In [ ]:
# Создадим признак BMI (Индекс массы тела) по формуле BMI = вес / рост^2
X_train_clean['bmi'] = X_train_clean['weight'] / (X_train_clean['height'] ** 2)
X_test['bmi'] = X_test['weight'] / (X_test['height'] ** 2)

In [ ]:
X_train_clean.head()

In [ ]:
# Создайте по такому же принципу еще несколько новых признаков
# Не забывайте создавать их как в X_train, так и в X_test
# Пишите ваш код в этой ячейке
...

### Этап 6: Кодирование категориальных признаков

In [ ]:
categorical_cols = ['gender', 'workout_type', 'favorite_color']

# Кодируем train и test отдельно
X_train_encoded = pd.get_dummies(X_train_clean, columns=categorical_cols, drop_first=True, dtype=int)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True, dtype=int)

# Выравниваем колонки
# Это нужно, чтобы число колонок в тесте и трейне после преобразований было одинаковым

all_columns = X_train_encoded.columns.union(X_test_encoded.columns) # Собирает все уникальные названия колонок из train и test
X_train_encoded = X_train_encoded.reindex(columns=all_columns, fill_value=0) # Добавляет в train недостающие колонки и заполняет их нулями
X_test_encoded = X_test_encoded.reindex(columns=all_columns, fill_value=0) # Добавляет в test недостающие колонки и заполняет их нулями

print(f"Train: {X_train_encoded.shape}, Test: {X_test_encoded.shape}")

In [ ]:
X_train_encoded.head(2)

In [ ]:
X_test_encoded.head(2)

### Этап 7: Масштабирование числовых признаков

In [ ]:
# Список числовых колонок
numerical_cols = ['Unnamed: 0','age', 'weight', 'height', 'avg_bpm', 'resting_bpm', 'session_duration', 'random_number', 'birth_year', 'bmi']

# Создаем копии трейна и теста
X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

# Масштабируем с помощью StandardScaler
scaler = MinMaxScaler()

# Обучаем scaler на трейне
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train_encoded[numerical_cols])

# Применяем к тесту
X_test_scaled[numerical_cols] = scaler.transform(X_test_encoded[numerical_cols])

print("Масштабирование завершено!")

In [ ]:
X_train_scaled.head(2)

In [ ]:
feature = 'weight' # Попробуйте посмотреть, как влияет масштабирование на разные признаки, меняя здесь название колонки

# Создаем два графика рядом
plt.figure(figsize=(12, 4))

# График ДО масштабирования
plt.subplot(1, 2, 1)
plt.hist(X_train_encoded[feature], bins=50, color='blue', alpha=0.7)
plt.title(f'ДО масштабирования\n{feature}')
plt.xlabel('Значение')
plt.ylabel('Количество')

# График ПОСЛЕ масштабирования
plt.subplot(1, 2, 2)
plt.hist(X_train_scaled[feature], bins=50, color='red', alpha=0.7)
plt.title(f'ПОСЛЕ масштабирования\n{feature}')
plt.xlabel('Значение после масштабирования')
plt.ylabel('Количество')

plt.tight_layout()
plt.show()

Обратите внимание на диапазон значений - после MinMaxScaler он будет от 0 до 1.

### Этап 8: Отбор признаков

Подумаем, какие колонки нужно удалить?
1. 'Unnamed: 0' - бесполезная колонка, дубликат id
2. 'random_number' - тоже бесполезная колонка, никак не относящаяся к теме
3. 'birth_year' - год рождения, дублирует возраст
4. 'favorite_color_Green', 'favorite_color_Purple', 'favorite_color_Red', 'favorite_color_Yellow' - любимый цвет никак не поможет нам в прогнозе

In [ ]:
# 1. Удалим явно бесполезные колонки
cols_to_drop = ['Unnamed: 0', 'random_number', 'birth_year', 'favorite_color_Green', 'favorite_color_Purple', 'favorite_color_Red', 'favorite_color_Yellow',]

X_train_final = X_train_scaled.drop(cols_to_drop, axis=1, errors='ignore')
X_test_final = X_test_scaled.drop(cols_to_drop, axis=1, errors='ignore')

In [ ]:
# 2. Построим матрицу корреляций и посмотрим, есть ли признаки с высокой корреляцией
plt.figure(figsize=(12, 8))
correlation_matrix = X_train_final.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Матрица корреляции признаков')
plt.tight_layout()
plt.show()

Если корреляция по модулю между двумя признаками > 0.9, можно удалить один из них.

### Итог

In [ ]:
# Проверка результатов
print("=== РЕЗУЛЬТАТЫ ПРЕДОБРАБОТКИ ===")
print(f"Исходный размер: {df.shape}")
print(f"Финальный размер train: {X_train_final.shape}")
print(f"Финальный размер test: {X_test_final.shape}")
print(f"Количество признаков: {X_train_final.shape[1]}")

# Проверка на пропуски
print(f"\nПропуски в train: {X_train_final.isnull().sum().sum()}")
print(f"Пропуски в test: {X_test_final.isnull().sum().sum()}")

# Проверка типов данных
print(f"\nТипы данных:")
print(X_train_final.dtypes.value_counts())

print("\n✅ Данные готовы для машинного обучения!")

### Запустим ML-модель на наших данных

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Простейшая линейная регрессия
model = LinearRegression()

# Обучаем на очищенных данных
model.fit(X_train_final, y_train_clean)

# Предсказываем
y_pred = model.predict(X_test_final)

# Считаем метрики
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("📊 РЕЗУЛЬТАТЫ МОДЕЛИ:")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}\n")


# Оценка результата
if r2 > 0.7:
    print("🔥 Отличный результат! Модель очень хорошо объясняет данные")
elif r2 > 0.5:
    print("✅ Хороший результат! Модель адекватно предсказывает калории") 
elif r2 > 0.3:
    print("⚠️ Средний результат. Есть куда улучшать")
elif r2 > 0:
    print("📉 Слабый результат. Нужна лучшая предобработка или фичи")
else:
    print("💥 Критично плохо. Проверь данные!")
